# Project 2 — Notebook 02: Cohort Month & Cohort Index

**Week 1 continued.** This notebook adds the three columns needed for retention analysis:

- `transaction_month`
- `cohort_month`
- `cohort_index`

## Step 1 — Import and load the clean dataset

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
OUTPUT_DIR = PROJECT_ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print('Project root:', PROJECT_ROOT)

import pandas as pd
import numpy as np

from src.data_prep import load_clean_dataset, add_cohort_columns

pd.set_option('display.max_columns', 30)

In [ ]:
df = load_clean_dataset(DATA_DIR)
print('clean rows      :', f'{len(df):,}')
print('unique customers:', f'{df["customer_id"].nunique():,}')
df.head(3)

## Step 2 — Add cohort columns

A cohort is a group of customers who started in the same month. Here, “started” means their first purchase month.

In [ ]:
df = add_cohort_columns(df)
df[['customer_id', 'invoice_datetime', 'transaction_month', 'cohort_month', 'cohort_index']].head(8)

## Step 3 — Validation checks

In [ ]:
print('Negative cohort_index rows:', (df['cohort_index'] < 0).sum())
print('Missing transaction_month :', df['transaction_month'].isna().sum())
print('Missing cohort_month      :', df['cohort_month'].isna().sum())
print('cohort_index range        :', df['cohort_index'].min(), '->', df['cohort_index'].max())

cohort_sizes = df.groupby('cohort_month')['customer_id'].nunique()
print('\nSum of cohort sizes      :', cohort_sizes.sum())
print('Unique customers overall :', df['customer_id'].nunique())
print('Do they match?           :', cohort_sizes.sum() == df['customer_id'].nunique())

## Step 4 — Cohort sizes

In [ ]:
cohort_sizes_df = cohort_sizes.rename('new_customers').to_frame()
cohort_sizes_df

## Step 5 — Save local cohort-ready data

In [ ]:
to_save = df.copy()
to_save['transaction_month'] = to_save['transaction_month'].astype(str)
to_save['cohort_month'] = to_save['cohort_month'].astype(str)
to_save.to_csv(DATA_DIR / 'cleaned_with_cohorts.csv', index=False)
cohort_sizes_df.to_csv(OUTPUT_DIR / 'cohort_sizes.csv')

print('Saved local file:', DATA_DIR / 'cleaned_with_cohorts.csv')
print('Saved cohort sizes:', OUTPUT_DIR / 'cohort_sizes.csv')

## Week 1 conclusion

The dataset now has a clear starting month for every customer and a `cohort_index` that counts Month 0, Month 1, Month 2, and so on. This is the base for the Week 2 retention matrix.